# Monthly data on deposits and casualties


In [48]:
import pandas as pd
from datetime import datetime


In [72]:
#load annual data 
data_cs = pd.read_csv('../data/intermediate/annual.csv')
data_cs = data_cs[data_cs["Year"] == 2022]
data_cs = data_cs[['Region', 'Excessive Marriages', 'ex_nup_per100k', '% Russians', 'population',
       'urban_share', 'uneployment', 'median_income',
       'consumption', 'av_income', 'share_poverty']]

#load political data
data_gov = pd.read_csv('../data/gov.csv')
today = pd.to_datetime(datetime.today().date())
data_gov['end'] = data_gov['end'].fillna(today)

#load monthly casualties 
df_monthly = pd.read_csv('../data/daily.csv')
df_monthly = df_monthly[['region', 'branch', 'name','slavic','death_month']]
df_monthly = df_monthly.rename(columns={df_monthly.columns[0] : "Region"})

#load monthly deposits
dep = pd.read_excel("../data/deposits.xlsx", header = 1)[:95]
dep = dep.rename(columns={dep.columns[0] : "Region"})
dep_long = pd.melt(dep, id_vars=['Region'],
                   var_name='Date', 
                   value_name='Deposits') 



In [73]:
#regional names unification

region_name_mapping = {
    'Архангельская область без данных по Ненецкому автономному округу': 'Архангельская область без АО',
    'в том числе Ненецкий автономный округ': 'Ненецкий АО',
    'Кемеровская область - Кузбасс': 'Кемеровская область',
    'Город Москва столица Российской Федерации город федерального значения': 'Москва',
    'Город Санкт-Петербург город федерального значения': 'Санкт-Петербург',
    'Город федерального значения Севастополь': 'Севастополь',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Карачаево-Черкесская Республика': 'Карачаево-Черкесия',
    'Республика Адыгея (Адыгея)': 'Республика Адыгея',
    'Республика Саха (Якутия)': 'Якутия',
    'Республика Северная Осетия - Алания': 'Северная Осетия',
    'Республика Татарстан (Татарстан)': 'Республика Татарстан',
    'Чувашская Республика - Чувашия': 'Чувашская Республика',
    'Тюменская область без данных по Ханты-Мансийскому автономному округу - Югре и Ямало-Ненецкому автономному округу': 'Тюменская область без АО',
    'в том числе Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский АО',
    'в том числе Ямало-Ненецкий автономный округ': 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
    'г. Москва' : 'Москва',
    'г. Санкт-Петербург' : 'Санкт-Петербург',
    'г. Севастополь' : 'Севастополь'
}

dep_long['Region'] = dep_long['Region'].replace(region_name_mapping)

In [74]:
#regional names unification

region_name_mapping = {
    'Архангельская область': 'Архангельская область без АО',
    'Ненецкий автономный округ': 'Ненецкий АО',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Республика Карачаево-Черкесия': 'Карачаево-Черкесия',
    'Республика Саха (Якутия)': 'Якутия',
    'Республика Северная Осетия-Алания': 'Северная Осетия',
    'Тюменская область': 'Тюменская область без АО',
    'Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский АО',
    'Ямало-Ненецкий автономный округ': 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
}

df_monthly['Region'] = df_monthly['Region'].replace(region_name_mapping)


In [75]:
#breaking down casualties by branch

contract = ['автомобильные', 'артиллерия', 'ВДВ', 'военмед', 'военные пилоты',
       'войска связи', 'войсковая ПВО', 'инженерные войска', 'МВД','морпехи', 'моряки',
       'мотострелковые войска', 'наземные авиаслужбы', 'нацгвардия',
       'РХБЗ', 'спецназ', 'танковые войска', 'ФСБ', 
       'другие войска', 'ЖД', 'военная полиция', 'СК', 'ФСО']

df_monthly["total"] = 1
df_monthly["drafted"] = df_monthly["pmc"] = df_monthly["volunteers"] = df_monthly["prisoners"] = df_monthly['contract'] = 0
df_monthly.loc[df_monthly['branch'] == 'добровольцы', 'volunteers'] = 1
df_monthly.loc[df_monthly['branch'] == 'мобилизованные', 'drafted'] = 1
df_monthly.loc[df_monthly['branch'] == 'ЧВК', 'pmc'] = 1
df_monthly.loc[df_monthly['branch'] == 'заключенные', 'prisoners'] = 1
df_monthly.loc[df_monthly['branch'].isin(contract), 'contract'] = 1

In [76]:
#date format unification

df_monthly['death_month'] = pd.to_datetime(df_monthly['death_month'].astype(str), format='%m.%Y', errors ='coerce')
dep_long['Date'] = pd.to_datetime(dep_long['Date'], format='%d.%m.%Y', errors='coerce')
data_gov['start'] = pd.to_datetime(data_gov['start'], errors='coerce')
data_gov['end'] = pd.to_datetime(data_gov['end'], errors='coerce')


df_agg = df_monthly.dropna(subset = ['death_month']).groupby(['Region', 'death_month']).agg(
    slavic_name = ('slavic', lambda x: (x == 1).sum()),
    non_slavic_name = ('slavic', lambda x: (x == 0).sum()),
    pmc = ('pmc', lambda x: (x == 1).sum()),
    drafted = ('drafted', lambda x: (x == 1).sum()),
    volunteers = ('volunteers', lambda x: (x == 1).sum()),
    prisoners = ('prisoners', lambda x: (x == 1).sum()),
    contract = ('contract', lambda x: (x == 1).sum()),
    total = ('total', lambda x: (x == 1).sum())).reset_index() 


In [78]:
all_regions = df_monthly['Region'].unique()
all_months = pd.date_range(
    start=df_monthly['death_month'].min(),
    end=df_monthly['death_month'].max(),
    freq='MS' 
)

complete_grid = pd.MultiIndex.from_product(
    [all_regions, all_months],
    names=['Region', 'death_month']
).to_frame(index=False)


In [79]:
final_result = (
    complete_grid.merge(
        df_agg,
        on=['Region', 'death_month'],
        how='left'
    )
    .fillna({'slavic_name': 0, 'non_slavic_name': 0})  
    .sort_values(['Region', 'death_month'])
)
final_result[['slavic_cumulative', 'non_slavic_cumulative']] = (
    final_result.groupby('Region')[['slavic_name', 'non_slavic_name']]
    .cumsum()
)


final_result.columns.values[1] = 'Date'

In [80]:
final_result = final_result.merge(dep_long,
                                  on = ['Region', 'Date'],
                                  how = 'left')
final_result = final_result.merge(data_cs, on = 'Region', how = 'left')
merged = final_result.merge(
    data_gov,
    on='Region',
    how='left'
)

# Filter rows where the date falls within the governor's term
merged = merged[
    (merged['Date'] >= merged['start']) &
    (merged['Date'] <= merged['end'])
]



In [82]:
merged.to_csv('../data/intermediate/monthly.csv')

In [89]:
df = final_result
df['excess_x_treat'] = df['Excessive Marriages per100k'] * df['treat']
df = df.rename(columns={'Excessive Marriages per100k': 'excess_marriage'})
df = df.rename(columns={'% Russians': 'share_rus'})

df_model = df[["Region", 'Date', "Deposits", "excess_x_treat", "treat", "excess_marriage", "total", 
               "unemp", "median_income", "share_poverty", "share_rus"]]
df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()

model1 = smf.ols(
    formula='Deposits ~ total + unemp + median_income + share_poverty + share_rus + excess_marriage + treat + excess_x_treat',
    data=df_model
).fit()

model1.summary()


C:\Users\Robert_Beta\AppData\Local\Temp\ipykernel_9208\1870795876.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               Deposits   R-squared:                       0.231
Model:                            OLS   Adj. R-squared:                  0.229
Method:                 Least Squares   F-statistic:                     126.0
Date:                 Сб, 26 июл 2025   Prob (F-statistic):          3.94e-185
Time:                        23:23:00   Log-Likelihood:                -52444.
No. Observations:                3360   AIC:                         1.049e+05
Df Residuals:                    3351   BIC:                         1.050e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept       -2.778e+06   2.87e+05     -9.682      0.000   -3.34e+06   -2.22e+06
total             532.8345     41.304     12.900      0.000     451.850     613.819
unemp           -6794.8077   1.13e+04     -0.603      0.546   -2.89e+04    1.53e+04
median_income      80.3276      3.455     23.250      0.000      73.554      87.102
share_poverty    4.937e+04   1.14e+04      4.335      0.000     2.7e+04    7.17e+04
share_rus        8642.0270   1645.405      5.252      0.000    5415.928    1.19e+04
excess_marriage -1563.2344    339.845     -4.600      0.000   -2229.560    -896.909
treat           -1.489e+05   1.14e+05     -1.312      0.190   -3.72e+05    7.37e+04
excess_x_treat   -113.5786    347.810     -0.327      0.744    -795.519     568.362
==============================================================================
Omnibus:                     4198.672   Durbin-Watson:                   0.064
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           638904.436
Skew:                           6.772   Prob(JB):                         0.00
Kurtosis:                      69.183   Cond. No.                     3.44e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.44e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [56]:
df_model.Region.unique()

array(['Алтайский край', 'Амурская область',
       'Архангельская область без АО', 'Астраханская область',
       'Белгородская область', 'Брянская область', 'Владимирская область',
       'Волгоградская область', 'Вологодская область',
       'Воронежская область', 'Еврейская АО', 'Забайкальский край',
       'Ивановская область', 'Иркутская область', 'Кабардино-Балкария',
       'Калининградская область', 'Калужская область', 'Камчатский край',
       'Карачаево-Черкесия', 'Кемеровская область', 'Кировская область',
       'Костромская область', 'Краснодарский край', 'Красноярский край',
       'Курганская область', 'Курская область', 'Ленинградская область',
       'Липецкая область', 'Магаданская область', 'Москва',
       'Московская область', 'Мурманская область', 'Ненецкий АО',
       'Нижегородская область', 'Новгородская область',
       'Новосибирская область', 'Омская область', 'Оренбургская область',
       'Орловская область', 'Пензенская область', 'Пермский край',
      

'Excessive Marriages per100k'